## Exercise 2 (1 point)
Implement read_remote_csv and read_remote_parquet tools that read a CSV/Parquet file from URL and return it to the LLM as text. Use Polars for this.

You can use any publicly hosted files, or make your own. Public examples:

CSV - our own ApisTox dataset)
Parquet - New York yellow taxi rides (select a few dozen rows for testing)
Test your LLM with the tool, asking a few questions about the dataset. First, you need to provide it with the URL to point to the file.

In [3]:
import io
import json
import os
from typing import Callable

import polars as pl
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

API_KEY = os.getenv("GEMINI_API_KEY")

# MODEL = "gemini-2.5-flash-lite"  # 20 req/day free tier
MODEL = "gemini-3.1-flash-lite-preview"  # test phase, lots of tokens


client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)


In [4]:
def make_llm_request(prompt: str) -> str:
    client = OpenAI(
        api_key=API_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )
    messages = [
        {"role": "developer", "content": "You are a weather assistant."},
        {"role": "user", "content": prompt},
    ]

    tool_definitions, tool_name_to_func = get_tool_definitions()

    # guard: loop limit, we break as soon as we get an answer
    for _ in range(10):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tool_definitions,  # always pass all tools in this example
            tool_choice="auto",
            max_completion_tokens=1000,
            # extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        resp_message = response.choices[0].message
        messages.append(
            {k: v for k, v in resp_message.model_dump().items() if v is not None}
        )

        print(f"Generated message: {resp_message.model_dump()}")
        print()

        # parse possible tool calls (assume only "function" tools)
        if resp_message.tool_calls:
            for tool_call in resp_message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                # call tool, serialize result, append to messages
                func = tool_name_to_func[func_name]
                func_result = func(**func_args)

                messages.append(
                    {
                        "role": "tool",
                        "content": json.dumps(func_result),
                        "tool_call_id": tool_call.id,
                    }
                )
        else:
            # no tool calls, we're done
            return resp_message.content

    # we should not get here
    last_response = resp_message.content
    return f"Could not resolve request, last response: {last_response}"


def get_tool_definitions() -> tuple[list[dict], dict[str, Callable]]:
    tool_definitions = [
        {
            "type": "function",
            "function": {
                "name": "read_remote_csv",
                "description": "Download a CSV file from a URL and return its contents as text.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {
                            "type": "string",
                            "description": "The URL of the CSV file to read.",
                        },
                    },
                    "required": ["url"],
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "read_remote_parquet",
                "description": "Download a Parquet file from a URL and return its contents as text.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {
                            "type": "string",
                            "description": "The URL of the Parquet file to read.",
                        },
                    },
                    "required": ["url"],
                },
            },
        },
    ]

    tool_name_to_callable = {
        "read_remote_csv": read_remote_csv_tool,
        "read_remote_parquet": read_remote_parquet_tool,
    }

    return tool_definitions, tool_name_to_callable


def read_remote_csv_tool(url: str) -> str:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    df = pl.read_csv(io.BytesIO(response.content))
    return df.head(20).write_csv()


def read_remote_parquet_tool(url: str) -> str:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    df = pl.read_parquet(io.BytesIO(response.content))
    return df.head(20).write_csv()


Csv call: 

In [5]:
prompt = "Based on returned data, tell me the domain of this dataset: https://github.com/j-adamczyk/ApisTox_dataset/blob/master/outputs/dataset_final.csv"
response = make_llm_request(prompt)
print("Prompt:\n", prompt)
print("Response:\n", response)


Generated message: {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'tYeahjgv', 'function': {'arguments': '{"url":"https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/refs/heads/master/outputs/dataset_final.csv"}', 'name': 'read_remote_csv'}, 'type': 'function', 'extra_content': {'google': {'thought_signature': 'EjQKMgEMOdbHQo8PsEwGpRVk7yL2NA8miDYWtI+Bhqd6ahEryJrhUgkF6kzqCzTExDV6d6Oo'}}}]}

Generated message: {'content': 'The dataset provided is in the domain of **toxicology**, specifically focused on **agrochemicals and their effects on bees (Apis toxicity)**.\n\nThe data consists of chemical compounds (identified by name, CID, CAS, and SMILES notation) along with information related to:\n*   **Toxicity/Exposure:** Classification of toxicity type (e.g., Contact, Oral).\n*   **Chemical Category:** Classification of the agrochemical (e.g., herbicide, fungicide, insecticide, or other).\n*   *

Parquet call: 

In [6]:
prompt = "Tell me something interesting about this dataset: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"
response = make_llm_request(prompt)
print("Prompt:\n", prompt)
print("Response:\n", response)

print()

Generated message: {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'TqZfE3Sd', 'function': {'arguments': '{"url":"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"}', 'name': 'read_remote_parquet'}, 'type': 'function', 'extra_content': {'google': {'thought_signature': 'EjQKMgEMOdbH9V0EHpqpFieZiUmNPCvy0qQdGYqb6X8nQmcBSmDiwevXB9PNAEId8oJgBUD2'}}}]}

Generated message: {'content': 'This dataset represents taxi trip data for New York City in **January 2026**.\n\nSomething interesting about this data is that it is a **"future" dataset**. Since we are currently in 2024, the inclusion of "2026-01" trip records suggests that this file may be a placeholder, a test file, or a synthetic projection provided by the TLC (Taxi & Limousine Commission) or the hosting platform for data modeling and software testing purposes.\n\nAdditionally, looking at the sample data provided:\n* **Ze

Parquet call 2: 

In [ ]:
for _ in range(1, 30):
    prompt = "Analyze pickup locations from this dataset. https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"
    response = make_llm_request(prompt)
    print("Prompt:\n", prompt)
    print("Response:\n", response)

    print()

UsageError: Cell magic `%%` not found.


## Running it as MCP server:

In [8]:
import asyncio
import json
from contextlib import AsyncExitStack
from openai import OpenAI
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


class MCPManager:
    def __init__(self, servers: dict[str, str]):
        self.servers = servers
        self.clients = {}
        self.tools = []  # in OpenAI format
        self._stack = AsyncExitStack()

    async def __aenter__(self):
        for url in self.servers.values():
            read, write, session_id = await self._stack.enter_async_context(
                streamable_http_client(url)
            )
            session = await self._stack.enter_async_context(ClientSession(read, write))
            await session.initialize()

            tools_resp = await session.list_tools()
            for t in tools_resp.tools:
                self.clients[t.name] = session
                self.tools.append(
                    {
                        "type": "function",
                        "function": {
                            "name": t.name,
                            "description": t.description,
                            "parameters": t.inputSchema,
                        },
                    }
                )

        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        await self._stack.aclose()

    async def call_tool(self, name: str, args: dict) -> dict | str:
        result = await self.clients[name].call_tool(name, arguments=args)
        return result.content[0].text


async def make_llm_request_mcp(prompt: str) -> str:
    mcp_servers = {
        "weather_forecast": "http://localhost:8001/mcp",
        "date_time_server": "http://localhost:8002/mcp",
    }

    gemini_client = OpenAI(
        api_key=API_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )

    async with MCPManager(mcp_servers) as mcp:
        messages = [
            {
                "role": "system",
                "content": "You are a helpful assistant. Use tools if you need to.",
            },
            {"role": "user", "content": prompt},
        ]

        for _ in range(10):
            response = gemini_client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=mcp.tools,
                tool_choice="auto",
                max_completion_tokens=1000,
            )

            resp_message = response.choices[0].message
            if not resp_message.tool_calls:
                return resp_message.content

            messages.append(
                {k: v for k, v in resp_message.model_dump().items() if v is not None}
            )
            for tool_call in resp_message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                print(f"Executing tool '{func_name}'")
                func_result = await mcp.call_tool(func_name, func_args)

                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": func_name,
                        "content": str(func_result),
                    }
                )


prompt = "What will be weather in Birmingham in two weeks?"
response = await make_llm_request_mcp(prompt)
print("Prompt:", prompt)
print("Response:", response)

print()

prompt = "What will be weather in Warsaw the day after tomorrow?"
response = await make_llm_request_mcp(prompt)
print("Prompt:", prompt)
print("Response:", response)

print()

prompt = "What is the current date and time?"
response = await make_llm_request_mcp(prompt)
print("Prompt:", prompt)
print("Response:", response)


CancelledError: Cancelled via cancel scope 15ef46bd0